In [7]:
import requests
import pandas as pd
import time
import datetime
import logging
import sqlite3


logging.basicConfig(
level=logging.INFO,#将日志存储加了个级别限制
format="%(asctime)s-%(levelname)s-%(message)s",#日志包含的具体消息
handlers=[
logging.FileHandler('spider_run_log',encoding='utf-8', mode="a"),logging.StreamHandler() ])#文件名字是spider_run_log，utf-8代表删除最后结果的逗号，a是让下一个数据直接加到后面，不用覆盖，文件夹和控制台都要显示
logging.getLogger("requests").setLevel(logging.WARNING)

#建立一个函数（get_best_descriptions）,目的是给的参数
def get_best_descriptions(descriptions,target_lang="en"):

    #如果没有描述，直接返回空列表
    if not descriptions:
       return ""

    #假设最开始的目标描述不存在
    target_desc = None

    #建立一个for 循环，当满足有中文叙述的条件，停止
    for goal in descriptions:
        if goal.get("lang") != target_lang:
           continue
        if not goal.get("value"):#这里的意思是我们得到描述goal存在内容的意思
           continue
        target_desc = goal.get("value")
        break

    #当没有没有目标描述的时候，取第一条描述就行了
    if not target_desc:
        target_desc = descriptions[0].get("value")

    #最后返还结果
    return target_desc or ""

url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
headers = {"User-Agent":"nvd-cve-collector/1.0",
           "apikey":""}

all_result = []
page = 1
max_page =4
start_index = 0
result_per_page = 50
max_retries = 2

#确定结束开始的时间，在datetime的箱子里找到datetime这个工具
today = datetime.date.today()
#利用timedelta指定一段时间
some_days_ago = today - datetime.timedelta(days = 7)
#将初始和结束的时间确定下来,strftime目的是为了格式化爬取的时间
pub_start_date = some_days_ago.strftime("%Y-%m-%dT00:00:00.000")
pub_end_date = today.strftime("%Y-%m-%dT23:59:59.999")


while True:
    logging.info(f"正在抓取{start_index//result_per_page + 1}页|抓去取了{start_index}条bugs|截至时间为{pub_end_date}")
    params ={
             "startIndex":start_index,
             "resultsPerPage":result_per_page,
             "pubStartDate":pub_start_date,
             "pubEndDate":pub_end_date
    }
    response = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                params = params,
                headers = headers,
                timeout = 50
            )

            #检查服务器返回时的HTTP的状态的状态码，将返还数据都编成Json的格式，方便读取
            response.raise_for_status()
            break

        #如果返回失败，检查具体原因
        except requests.exceptions.Timeout:
            logging.error(f"请求超时，第{attempt + 1}次重试")
        except requests.exceptions.ConnectionError as error:
            logging.error(f"连接错误:{error},第{attempt + 1}次重试")
        except requests.exceptions.RequestException as error:
            logging.error(f"请求失败：{error},第{attempt + 1}次重试")
        if attempt < max_retries - 1:
            time.sleep(2)


    else:
        logging.error("重试还是失败，所以放弃这一页")
        break

    if response.status_code == 200:#这表示服务器访问顺利
        try:
            data = response.json()
            #将返回的数据里的所有漏洞放进bugs这个列表
            bugs = data.get("vulnerabilities",[])

            if not bugs:
                break

            #将所有漏洞都循环一遍
            for bug in bugs:
                bug_datas = bug.get("cve",{})#如果存在漏洞，取出第一个漏洞，并且命名为bug_bags
                bug_name = bug_datas.get("id","")

                metrics = bug_datas.get("metrics",{})#将bug_bags这个漏洞所有信息放进metrics这个列表里面
                #将漏洞cvss最开始的数据设为0.0或者没有
                cvss_score = None
                cvss_version = None

                cvss_types = [
                    "cvssMetricV40",
                    "cvssMetricV31",
                    "cvssMetricV30",
                    "cvssMetricV2",
                ]
                #用以上的评分系统进行抓取信息，将评分系统在漏洞信息列表得出的所需数据做成一个新的列表
                for cvss_true_type in cvss_types:
                    metrics_list = metrics.get(cvss_true_type,[])

                    #取评分系统的第一版所需数据
                    if metrics_list:
                        metrics_data = metrics_list[0].get("cvssData",{})
                        cvss_score = metrics_data.get("baseScore")
                        cvss_version = metrics_data.get("version")
                        break

                    #提出描述
                descriptions = bug_datas.get("descriptions",[])
                target_desc = get_best_descriptions(descriptions,target_lang="en")

                all_result.append({
                    "cve_id":bug_datas.get("id",bug_name),
                    "source":"NVD",
                    "descriptions":target_desc or "",
                    "cvss_score":cvss_score,
                    "cvss_version":cvss_version or "",
                    "pubStartDate":pub_start_date,
                    "pubEndDate":pub_end_date
                })
            if len(bugs) < result_per_page or page >= max_page:
                break
            start_index += result_per_page
            page += 1
            time.sleep(2)

        except ValueError as error:
            logging.error("无法将数据统计为Json格式")
            break

    else:
        logging.error(f"服务器被拒绝访问或者服务器错误，状态码：{response.status_code}")
        break

#把所有结果都做成列表，df是数据表格缩写
df = pd.DataFrame(all_result)
#由于多次和多数漏洞的筛查，所有制作一个智能的垃圾桶，将不符合和重复的数据全都丢进垃圾桶,drop是扔掉的意思，duplicates是重复的东西的意思，subset是子集的意思，inplace表示直接在原表格上修改
df.drop_duplicates(subset="cve_id",inplace=True)
df.to_csv(
    "nvd_result.csv",
    index=False,
    encoding="utf-8-sig")#确保excel可以顺利打开,这是给人看的

#建立一个用sqlite表示的表格，给程序的看的,为了方便我理解 这里给名字设置成vulnerabilities
conn = sqlite3.connect("nvd_result.db")
df.to_sql("vulnerabilities",conn,if_exists="replace",index=False)
conn.close()

logging.info(f"今天的工作已完成了,哈哈哈哈哈，我们抓到了{len(all_result)},最终我们抓到并保存的有{len(df)},时间从{pub_start_date}至{pub_end_date}")

logging.info("抓取并清洗成功！")
print(df.to_string(index=False))



2026-09-22 18:47:41,613-INFO-正在抓取1页|抓去取了0条bugs|截至时间为2026-09-22T23:59:59.999
2026-09-22 18:47:46,702-INFO-正在抓取2页|抓去取了50条bugs|截至时间为2026-09-22T23:59:59.999
2026-09-22 18:47:50,430-INFO-正在抓取3页|抓去取了100条bugs|截至时间为2026-09-22T23:59:59.999
2026-09-22 18:47:53,672-INFO-正在抓取4页|抓去取了150条bugs|截至时间为2026-09-22T23:59:59.999
2026-09-22 18:47:56,765-INFO-今天的工作已完成了,哈哈哈哈哈，我们抓到了200,最终我们抓到并保存的有200,时间从2026-09-15T00:00:00.000至2026-09-22T23:59:59.999
2026-09-22 18:47:56,766-INFO-抓取并清洗成功！


        cve_id source                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   